# pipeline de preprocessing :



## 1. Imports et Chargement des donées

In [ ]:
# Import
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.preprocessing import LabelEncoder

# Charger le DataFrame nettoyé
df_clean = pd.read_csv("../../data/processed/pokemon_clean.csv")

# On crée une copie du DataFrame pour l'encodage
df_encoded = df_clean.copy()
df_encoded.head()

,Number,Name,Type 1,Type 2,Abilities,HP,Att,Def,Spa,Spd,...,Against Bug,Against Rock,Against Ghost,Against Dragon,Against Dark,Against Steel,Against Fairy,Height,Weight,BMI
0,1,Bulbasaur,Grass,Poison,"['Chlorophyll', 'Overgrow']",45,49,49,65,65,...,1.0,1.0,1.0,1.0,1.0,1.0,0.5,0.7,6.9,14.1
1,2,Ivysaur,Grass,Poison,"['Chlorophyll', 'Overgrow']",60,62,63,80,80,...,1.0,1.0,1.0,1.0,1.0,1.0,0.5,1.0,13.0,13.0
2,3,Venusaur,Grass,Poison,"['Chlorophyll', 'Overgrow']",80,82,83,100,100,...,1.0,1.0,1.0,1.0,1.0,1.0,0.5,2.0,100.0,25.0
3,3,Mega Venusaur,Grass,Poison,['Thick Fat'],80,100,123,122,120,...,1.0,1.0,1.0,1.0,1.0,1.0,0.5,2.4,155.5,27.0
4,4,Charmander,Fire,Fire,"['Blaze', 'Solar Power']",39,52,43,60,50,...,0.5,2.0,1.0,1.0,1.0,0.5,0.5,0.6,8.5,23.6


## 2. Renommer les colonnes

In [ ]:
df_encoded.columns = df_encoded.columns.str.lower().str.replace(' ', '_')
print(df_encoded.columns.tolist())  # vérifier le résultat

## 3. Supprimer les colonnes inutiles

- **number** : Simple identifiant du Pokédex, aucun lien avec le statut Légendaire.

- **name** : Identifiant textuel, le modèle ne peut pas exploiter un nom propre.

- **abilities** : certaines abilities sont exclusives aux Légendaires (ex: Pressure, Multiscale). Le modèle "tricherait" en les reconnaissant au lieu d'apprendre depuis les stats.

- **mega_evolution** : Une Méga-Évolution n'est pas un Légendaire mais a des stats énormes similaires. Garder cette colonne confondrait le modèle.

- **alolan_form / galarian_form** : Même problème — ce sont des formes alternatives de Pokémon normaux avec des stats modifiées, pas des Légendaires.

- **bmi** : Déjà présent dans le dataset original mais peu pertinent — il n'existe pas de logique biologique cohérente pour le BMI des Pokémon. De plus height et weight sont déjà présents comme features indépendantes.

In [ ]:
cols_to_drop = [
    'number',
    'name',
    'abilities',
    'mega_evolution',
    'alolan_form',
    'galarian_form',
    'bmi'           
]

df_encoded = df_encoded.drop(columns=cols_to_drop)
print(df_encoded.columns.tolist())  # vérifier le résultat

## 4. Encodage des variables 
On vas transformer les variables de chaine de caractere en nombre comprehensible par nos futurs model de machine learning.

Pour ca on va creer deux autre colones encodé 'type1_enc' et 'type2_enc' avec :
- ``LabelEncoder`` :  attribue un numéro à chaque catégorie unique
- ``fit_transform`` : permet la tranformation en chiffre

In [ ]:
# Encoder type1 et type2
encodeur = LabelEncoder()
df_encoded['type1_enc'] = encodeur.fit_transform(df_encoded['type1'].astype(str))
df_encoded['type2_enc'] = encodeur.fit_transform(df_encoded ['type2'].astype(str))



KeyError: 'type1'

## 5. Ajout de nouvelles colonnes

On ajoute des colones pertiante pour que nos model soit plus efficace :

On cherche les légendaire, 
1. ``has_type2`` : Or les légendaires ont souvent 2 types ! 
2. ``ratio_off_def``:  Et on souvent un profil combat ! 

In [ ]:
# 1. A-t-il un second type ?
# Quand on a nettoyé les données, on a mis Type 2 = Type 1 pour les pokémons sans second type
# Donc si Type 1 == Type 2, c'est qu'il n'avait pas de vrai second type
df_encoded['has_type2'] = (df_encoded['Type 1'] != df_encoded['Type 2']).astype(int)

# 2. Ratio Offensif / Défensif
df_encoded['ratio_off_def'] = (df_encoded['Att'] + df_encoded['Spa']) / (df_encoded['Def'] + df_encoded['Spd'])

## 6. Séparation des données

On separe les données du DataFrame en deux catégories :
- **X** : Toutes les colonnes (les statistiques)  qui servent à prédire
- **Y** : Ce qu'on veut prédire, c'est a dire estce que c'est un legendaire ?

Puis on les sépare en données d'entraînement et de test :
- **X_train / y_train** : Ce sont les données  d'entrainement
- **X_test / y_test** : Ce sont les données sur laquelle le modeles sera évalué

In [ ]:


# Toute les autres colonnes 
X = df.drop(columns=['Legendary'])

# Séparation en données d'entraînement et de test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

 
def prepare_data(df, target_col):

    """

    Split + encodage + normalisation via make_pipeline.

    Entrée : dataframe et colonne cible

    Retourne : X_train, X_test, y_train, y_test, pipeline complet

    """

    # On sépare la colone cible des autres colonnes
    # La colonne cible
    y = df['Legendary']

    y = df[target_col]

    cat_cols = X.select_dtypes(include='object').columns

    num_cols = X.select_dtypes(include=['int64','float64']).columns

    preprocessor = ColumnTransformer([

        ('num', make_pipeline(StandardScaler()), num_cols),

        ('cat', make_pipeline(OneHotEncoder(sparse=False, handle_unknown='ignore')), cat_cols)

    ])

    X_train, X_test, y_train, y_test = train_test_split(X, y)

    X_train = preprocessor.fit_transform(X_train)

    X_test = preprocessor.transform(X_test)

    return X_train, X_test, y_train, y_test, preprocessor
 

``random_state=42`` : les données seront toujours mélanger de la meme facon, pour qu'on obtienne les meme resultat a chque réexécution

``stratify=y`` : garantit les mêmes proportions de légendaires dans X_train/y_train 
et X_test/y_test — indispensable car les légendaires sont peu nombreux dans le dataset ! 